<a href="https://colab.research.google.com/github/KyberNull/embeddinggemma2_asr/blob/main/DSP_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EmbeddingGemma 2 ASR — Full Training v3 (Fixed Memory Leak + TPU + Drive Auto-Resume)

Fixes the host RAM growth / slowdown (`1.11s/it` → `11.23s/it` → OOM) by bounding TPU/XLA graph compilations to **5 fixed audio buckets** (4s, 8s, 12s, 16s, 20s) and keeping intermediate 12-layer stacks in `bfloat16`.

Automatically resumes from `/content/drive/MyDrive/DSP_Mini_Project_ASR/latest_ckpt.pt` without losing your Epoch 4 progress (`val_cer = 0.0987`).

## 1. Install dependencies with `uv`

In [ ]:
!command -v uv >/dev/null 2>&1 || pip -q install uv
!uv pip install --system -q torch_tpu polars datasets
!uv pip install --system -q -U "transformers>=5.19.0" librosa soundfile tqdm jiwer regex

## 2. Mount Google Drive & verify TPU

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/DSP_Mini_Project_ASR"
os.makedirs(DRIVE_DIR, exist_ok=True)
LATEST_CKPT = os.path.join(DRIVE_DIR, "latest_ckpt.pt")
BEST_CKPT = os.path.join(DRIVE_DIR, "best_model.pt")
VOCAB_PATH = os.path.join(DRIVE_DIR, "vocab.json")
HISTORY_PATH = os.path.join(DRIVE_DIR, "history.csv")

print("Drive checkpoint dir:", DRIVE_DIR)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("TPU available:", torch.accelerator.is_available(), "| Count:", torch.accelerator.device_count())

if not torch.accelerator.is_available():
    raise RuntimeError("No TPU detected. Choose Runtime > Change runtime type > TPU.")

DEVICE = torch.device("tpu")
x = torch.randn(128, 128, device=DEVICE)
_ = x @ x
torch.accelerator.synchronize()
print("TPU ready:", DEVICE)

## 3. Training configuration & vocabulary

In [ ]:
import json, unicodedata, regex

DATASET_ID = "openslr/librispeech_asr"
DATASET_CONFIG = "clean"
SAMPLE_RATE = 16000

MAX_TRAIN_UTTERANCES = None    # full train.100 (~28,539 utterances)
MAX_VAL_UTTERANCES = 500       # fast validation per epoch

BATCH_SIZE = 16
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4

# Fixed bucketing caps XLA graph cache to 5 shapes -> flat RAM usage
AUDIO_BUCKET_SEC = 4           # pad raw audio to 4s, 8s, 12s, 16s, 20s
MAX_AUDIO_SEC = 20             # skip rare >20s outliers
BUCKET_STEP = 64               # pad encoder T to multiples of 64
SAVE_EVERY_STEPS = 250         # checkpoint to Drive every N steps
SHUFFLE_BUFFER = 128           # lower host RAM footprint

BLANK_ID = 0
itos = ["<blank>", " ", "'"] + [chr(c) for c in range(ord("A"), ord("Z") + 1)]
stoi = {tok: i for i, tok in enumerate(itos)}

def normalize_text(s):
    return unicodedata.normalize("NFC", str(s).strip().upper())

def graphemes(s):
    return regex.findall(r"\X", normalize_text(s))

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(itos, f, ensure_ascii=False, indent=2)

print(f"Vocab size: {len(itos)} | Saved to {VOCAB_PATH}")

## 4. Load frozen EmbeddingGemma 2 audio tower (304.8M Conformer)

In [ ]:
import gc, inspect
from transformers import AutoProcessor, AutoModel

MODEL_ID = "google/embeddinggemma-2"

processor = AutoProcessor.from_pretrained(MODEL_ID)
full_model = AutoModel.from_pretrained(
    MODEL_ID,
    vision_config=None,
    dtype=torch.bfloat16,
)

audio_tower = full_model.audio_tower.to(DEVICE)
audio_tower.eval()
for p in audio_tower.parameters():
    p.requires_grad_(False)

del full_model
gc.collect()

_TOWER_SIG = inspect.signature(audio_tower.forward).parameters
n_params = sum(p.numel() for p in audio_tower.parameters())
print(f"Loaded {type(audio_tower).__name__} ({n_params / 1e6:.1f}M params) on {next(audio_tower.parameters()).device}")

## 5. Fixed-bucket streaming batch extractor (prevents XLA cache growth)

Pads raw waveforms to fixed multiples of `4s` before calling `processor` so `audio_tower` only ever compiles 5 static input shapes. Keeps the 12-layer feature stack in `bfloat16` to halve TPU memory.

In [ ]:
import itertools, math, librosa
import numpy as np
import torch.nn.functional as F
from datasets import load_dataset

def prepare_audio(audio_dict):
    wav = np.asarray(audio_dict["array"], dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    sr = audio_dict["sampling_rate"]
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    return wav

@torch.inference_mode()
def encode_batch(wavs):
    raw_lens = [len(w) for w in wavs]
    max_raw = max(raw_lens)
    bucket_samples = int(
        math.ceil(max_raw / (AUDIO_BUCKET_SEC * SAMPLE_RATE)) * AUDIO_BUCKET_SEC * SAMPLE_RATE
    )

    # Fixed-length NumPy padding -> only 5 unique shapes for audio_tower
    padded_wavs = [np.pad(w, (0, bucket_samples - len(w))) for w in wavs]
    batch = processor(
        audio=padded_wavs,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    )

    input_features = batch["input_features"].to(DEVICE, dtype=torch.bfloat16)
    mask = batch.get("input_features_mask")
    if mask is None:
        mask = batch.get("attention_mask")

    kwargs = {"output_hidden_states": True, "return_dict": True}
    if mask is not None:
        mask_dev = mask.to(DEVICE)
        if "input_features_mask" in _TOWER_SIG:
            kwargs["input_features_mask"] = mask_dev
        elif "attention_mask" in _TOWER_SIG:
            kwargs["attention_mask"] = mask_dev

    out = audio_tower(input_features, **kwargs)
    if isinstance(out, tuple):
        hs = (out[0],)
    else:
        hs = getattr(out, "hidden_states", None) or (out.last_hidden_state,)

    # Keep only the 12 Conformer layers (1024-D); stay in bfloat16 to halve memory
    base_dim = hs[0].shape[-1]
    layers = [h for h in hs if h.shape[-1] == base_dim]
    feats = torch.stack(layers, dim=1).to(torch.bfloat16)  # [B, L, T, D]
    B, L, T, D = feats.shape

    ratios = torch.tensor([r / bucket_samples for r in raw_lens], dtype=torch.float32)
    out_lens = torch.clamp((ratios * T).round().long(), min=1, max=T)

    padded_T = int(math.ceil(T / BUCKET_STEP) * BUCKET_STEP)
    if padded_T > T:
        feats = F.pad(feats, (0, 0, 0, padded_T - T))

    return feats, out_lens

def stream_batches(split, batch_size, max_utterances=None, shuffle=False, seed=42, drop_last=True):
    ds = load_dataset(DATASET_ID, DATASET_CONFIG, split=split, streaming=True)
    if shuffle:
        ds = ds.shuffle(seed=seed, buffer_size=SHUFFLE_BUFFER)
    iterator = ds if max_utterances is None else itertools.islice(ds, max_utterances)

    wavs, target_list, texts = [], [], []
    for item in iterator:
        text = normalize_text(item["text"])
        if not text:
            continue
        toks = graphemes(text)
        if any(t not in stoi for t in toks):
            continue
        wav = prepare_audio(item["audio"])
        if len(wav) > MAX_AUDIO_SEC * SAMPLE_RATE:
            continue
        wavs.append(wav)
        target_list.append(torch.tensor([stoi[t] for t in toks], dtype=torch.long))
        texts.append(text)

        if len(wavs) == batch_size:
            feats, out_lens = encode_batch(wavs)
            yield feats, out_lens, target_list, texts
            wavs, target_list, texts = [], [], []

    if wavs and not drop_last:
        feats, out_lens = encode_batch(wavs)
        yield feats, out_lens, target_list, texts

# Smoke test one streamed batch
test_feats, test_lens, test_tgts, test_texts = next(
    stream_batches("validation", batch_size=2, max_utterances=2, drop_last=False)
)
N_LAYERS, INPUT_DIM = test_feats.shape[1], test_feats.shape[3]
print(f"Streamed batch shape: {tuple(test_feats.shape)} ({test_feats.dtype}) | lengths: {test_lens.tolist()}")

## 6. TPU CTC Reader (Learned Layer Mix + Causal Conv1D + Masked BiGRU)

In [ ]:
import torch.nn as nn

class TPUCTCReader(nn.Module):
    def __init__(self, n_layers, input_dim, vocab_size, hidden=256, gru_layers=2, kernel_size=5):
        super().__init__()
        self.layer_logits = nn.Parameter(torch.zeros(n_layers))
        self.norm_in = nn.LayerNorm(input_dim)
        self.proj = nn.Linear(input_dim, hidden)

        self.dwconv = nn.Conv1d(
            hidden, hidden, kernel_size=kernel_size,
            groups=hidden, padding=0,
        )
        self.conv_act = nn.GELU()

        self.gru = nn.GRU(
            input_size=hidden,
            hidden_size=hidden // 2,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.1 if gru_layers > 1 else 0.0,
        )
        self.norm_out = nn.LayerNorm(hidden)
        self.classifier = nn.Linear(hidden, vocab_size)
        self.kernel_size = kernel_size

    def forward(self, x, input_lengths=None):
        # x: [B, layers, T, D] (bfloat16)
        weights = torch.softmax(self.layer_logits, dim=0).to(x.dtype)
        x = torch.einsum("l,bltd->btd", weights, x).float()

        mask = None
        if input_lengths is not None:
            T = x.shape[1]
            mask = (
                torch.arange(T, device=x.device)[None, :] < input_lengths.to(x.device)[:, None]
            ).unsqueeze(-1)

        x = self.proj(self.norm_in(x))
        if mask is not None:
            x = x * mask

        x = x.transpose(1, 2)
        x = F.pad(x, (self.kernel_size - 1, 0))  # causal left padding
        x = self.dwconv(x)
        x = self.conv_act(x).transpose(1, 2)
        if mask is not None:
            x = x * mask

        x, _ = self.gru(x)
        if mask is not None:
            x = x * mask

        x = self.norm_out(x)
        logits = self.classifier(x)
        return logits.log_softmax(dim=-1)

reader = TPUCTCReader(
    n_layers=N_LAYERS,
    input_dim=INPUT_DIM,
    vocab_size=len(itos),
).to(DEVICE)

n_reader_params = sum(p.numel() for p in reader.parameters())
print(f"TPUCTCReader initialized ({n_reader_params / 1e6:.2f}M trainable params) on {DEVICE}")

## 7. Full training loop with Drive auto-resume & periodic GC

In [ ]:
import torch.optim as optim
import polars as pl
from jiwer import cer, wer
from tqdm.auto import tqdm

ctc_loss_fn = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)
optimizer = optim.AdamW(reader.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# --- Probe native TPU CTCLoss forward + backward ---
CTC_ON_TPU = False
try:
    _lp = torch.randn(32, 2, len(itos), device=DEVICE, requires_grad=True).log_softmax(dim=-1)
    _tgts = torch.tensor([1, 2, 3, 4], dtype=torch.long, device=DEVICE)
    _il = torch.tensor([32, 24], dtype=torch.long, device=DEVICE)
    _tl = torch.tensor([2, 2], dtype=torch.long, device=DEVICE)
    _loss = ctc_loss_fn(_lp, _tgts, _il, _tl)
    _loss.backward()
    torch.accelerator.synchronize()
    del _lp, _tgts, _il, _tl, _loss
    CTC_ON_TPU = True
    print("CTCLoss probe: native TPU CTCLoss succeeded (fast path enabled).")
except Exception as e:
    print(f"CTCLoss probe: TPU CTCLoss failed ({e!r}); using CPU CTCLoss fallback.")

def compute_ctc_loss(log_probs, flat_targets, input_lengths, target_lengths):
    lp_tbc = log_probs.transpose(0, 1).float()
    if CTC_ON_TPU:
        return ctc_loss_fn(
            lp_tbc,
            flat_targets.to(DEVICE),
            input_lengths.to(DEVICE),
            target_lengths.to(DEVICE),
        )
    return ctc_loss_fn(lp_tbc.cpu(), flat_targets, input_lengths, target_lengths)

def ctc_greedy_decode(frame_ids):
    out = []
    prev = None
    for idx in frame_ids:
        idx = int(idx)
        if idx != prev and idx != BLANK_ID:
            out.append(itos[idx])
        prev = idx
    return "".join(out)

def filter_ctc_valid(feats, out_lens, target_list, texts):
    keep = []
    for i, y in enumerate(target_list):
        ids = y.tolist()
        repeats = sum(ids[j] == ids[j - 1] for j in range(1, len(ids)))
        if int(out_lens[i]) >= len(ids) + repeats:
            keep.append(i)
    if not keep:
        return None
    if len(keep) == len(target_list):
        # Zero-copy fast path keeps static batch size B
        return (
            feats,
            out_lens,
            torch.cat(target_list),
            torch.tensor([len(y) for y in target_list], dtype=torch.long),
            texts,
        )
    idx = torch.tensor(keep, dtype=torch.long)
    return (
        feats[idx.to(feats.device)],
        out_lens[idx],
        torch.cat([target_list[i] for i in keep]),
        torch.tensor([len(target_list[i]) for i in keep], dtype=torch.long),
        [texts[i] for i in keep],
    )

def save_checkpoint(path, epoch, global_step, best_val_cer, history):
    ckpt = {
        "model": {k: v.detach().cpu() for k, v in reader.state_dict().items()},
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "itos": itos,
        "input_dim": INPUT_DIM,
        "n_layers": N_LAYERS,
        "epoch": epoch,
        "global_step": global_step,
        "best_val_cer": best_val_cer,
        "history": history,
    }
    tmp_path = path + ".tmp"
    torch.save(ckpt, tmp_path)
    os.replace(tmp_path, path)

# Auto-resume from Drive if checkpoint exists
start_epoch = 0
global_step = 0
best_val_cer = float("inf")
history = []

if os.path.exists(LATEST_CKPT):
    print("Resuming from Drive checkpoint:", LATEST_CKPT)
    ckpt = torch.load(LATEST_CKPT, map_location="cpu", weights_only=False)
    reader.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"]
    global_step = ckpt.get("global_step", 0)
    best_val_cer = ckpt.get("best_val_cer", float("inf"))
    history = ckpt.get("history", [])
    del ckpt
    gc.collect()
    print(f"Resumed at epoch {start_epoch}, step {global_step}, best_val_cer={best_val_cer:.4f}")

@torch.no_grad()
def validate():
    reader.eval()
    losses, refs, hyps = [], [], []
    for feats, out_lens, target_list, texts in stream_batches(
        "validation", batch_size=BATCH_SIZE, max_utterances=MAX_VAL_UTTERANCES, shuffle=False, drop_last=False
    ):
        filtered = filter_ctc_valid(feats, out_lens, target_list, texts)
        if filtered is None:
            continue
        x, input_lengths, flat_targets, target_lengths, kept_texts = filtered
        lp = reader(x, input_lengths)
        loss = compute_ctc_loss(lp, flat_targets, input_lengths, target_lengths)
        losses.append(float(loss.detach().cpu()))

        preds = lp.cpu().argmax(dim=-1)
        for i in range(preds.shape[0]):
            refs.append(kept_texts[i])
            hyps.append(ctc_greedy_decode(preds[i, :input_lengths[i]].tolist()))
        del feats, x, lp, loss

    gc.collect()
    return float(np.mean(losses)), float(cer(refs, hyps)), float(wer(refs, hyps))

for epoch in range(start_epoch, EPOCHS):
    reader.train()
    losses = []
    pbar = tqdm(
        stream_batches(
            "train.100",
            batch_size=BATCH_SIZE,
            max_utterances=MAX_TRAIN_UTTERANCES,
            shuffle=True,
            seed=42 + epoch,
            drop_last=True,
        ),
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )
    for feats, out_lens, target_list, texts in pbar:
        filtered = filter_ctc_valid(feats, out_lens, target_list, texts)
        if filtered is None:
            del feats
            continue
        x, input_lengths, flat_targets, target_lengths, _ = filtered

        optimizer.zero_grad(set_to_none=True)
        log_probs = reader(x, input_lengths)
        loss = compute_ctc_loss(log_probs, flat_targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reader.parameters(), 5.0)
        optimizer.step()

        loss_val = float(loss.detach().cpu())
        losses.append(loss_val)
        global_step += 1
        pbar.set_postfix(ctc=f"{loss_val:.3f}", step=global_step)

        del feats, x, log_probs, loss
        if global_step % 50 == 0:
            gc.collect()
        if global_step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(LATEST_CKPT, epoch, global_step, best_val_cer, history)

    scheduler.step()
    tr_loss = float(np.mean(losses))
    va_loss, va_cer, va_wer = validate()

    row = {
        "epoch": epoch + 1,
        "step": global_step,
        "lr": float(scheduler.get_last_lr()[0]),
        "train_ctc": tr_loss,
        "val_ctc": va_loss,
        "val_cer": va_cer,
        "val_wer": va_wer,
    }
    history.append(row)
    print(row)

    save_checkpoint(LATEST_CKPT, epoch + 1, global_step, min(best_val_cer, va_cer), history)
    if va_cer < best_val_cer:
        best_val_cer = va_cer
        save_checkpoint(BEST_CKPT, epoch + 1, global_step, best_val_cer, history)
        print(f"  -> New best model saved to Drive ({BEST_CKPT}) | val_cer={best_val_cer:.4f}")

    pl.DataFrame(history).write_csv(HISTORY_PATH)
    gc.collect()

pl.DataFrame(history)

## 8. Inspect best model predictions & learned layer weights

In [ ]:
best = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
reader.load_state_dict(best["model"])
reader.eval()

weights = torch.softmax(reader.layer_logits.detach().cpu(), dim=0).tolist()
print("Best epoch:", best["epoch"], "| Best val CER:", round(best["best_val_cer"], 4))
print("Learned layer mix weights:", [round(w, 4) for w in weights])
print("=" * 60)

@torch.no_grad()
def inspect_predictions(n=10):
    shown = 0
    for feats, out_lens, target_list, texts in stream_batches(
        "validation", batch_size=BATCH_SIZE, max_utterances=n, shuffle=False, drop_last=False
    ):
        lp = reader(feats, out_lens).cpu()
        pred = lp.argmax(dim=-1)
        for i in range(pred.shape[0]):
            hyp = ctc_greedy_decode(pred[i, :out_lens[i]].tolist())
            print("REF:", texts[i])
            print("HYP:", hyp)
            print("-" * 60)
            shown += 1
            if shown >= n:
                return

inspect_predictions()